# 05 — MLP models

MLP classifier training and evaluation. Uses `src.models.mlp` and `src.training.evaluation`.

In [ ]:
# Path fix: use this repo's src
from pathlib import Path
import sys
import logging
import numpy as np
import torch
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils import SEED, set_global_seed
from src.utils.paths import get_reduced_dir, get_splits_dir, get_labels_dir, get_experiment_dir, get_experiment_output_dir, ensure_dir
from src.data.splits import load_splits
from src.data.io import load_labels
from src.models.mlp import MLPClassifier
from src.training.evaluation import compute_metrics, save_metrics, save_confusion_matrix

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# MLP / training params (match A1 and B1 configs)
EPOCHS = 20
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIMS = [512, 256]
DROPOUT = 0.1
NUM_CLASSES = 4
CLASS_NAMES = ["class_0", "class_1", "class_2", "class_3"]
print("MLP: epochs=%s batch=%s lr=%s" % (EPOCHS, BATCH_SIZE, LR))

In [ ]:
# Load splits and labels (same order as saved train/test/val .npy)
print("Loading splits and labels...")
idx_train, idx_val, idx_test = load_splits()
try:
    y_full = load_labels()
except FileNotFoundError:
    print("labels.npy not found; loading from raw dataset...")
    from src.data import load_raw_dataset
    _, y_full = load_raw_dataset()
if y_full.ndim == 2:
    y_idx = np.argmax(y_full, axis=1)
else:
    y_idx = y_full
y_train = y_idx[idx_train]
y_test = y_idx[idx_test]
y_val = y_idx[idx_val] if len(idx_val) > 0 else None
print("y_train %s y_test %s" % (y_train.shape, y_test.shape))

In [ ]:
def train_mlp_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, input_dim, condition_name):
    """Train MLP, save checkpoint, evaluate on test, save metrics and confusion matrix."""
    set_global_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    exp_dir = ensure_dir(get_experiment_dir(condition_name))
    ckpt_dir = ensure_dir(get_experiment_output_dir(condition_name, checkpoints=True))

    model = MLPClassifier(
        input_dim=input_dim,
        num_classes=NUM_CLASSES,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    n_train = len(X_train)

    for ep in range(EPOCHS):
        model.train()
        perm = np.random.permutation(n_train)
        total_loss = 0.0
        n_b = 0
        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start : start + BATCH_SIZE]
            bx = torch.from_numpy(X_train[idx]).float().to(device)
            by = torch.from_numpy(y_train[idx]).long().to(device)
            logits = model(bx)
            loss = torch.nn.functional.cross_entropy(logits, by)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_b += 1
        if (ep + 1) % 5 == 0 or ep == 0:
            print("  Epoch %d loss %.4f" % (ep + 1, total_loss / max(n_b, 1)))

    torch.save(model.state_dict(), ckpt_dir / "mlp.pt")
    print("  Saved %s" % (ckpt_dir / "mlp.pt"))

    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_test).float().to(device))
        y_pred = logits.argmax(dim=1).cpu().numpy()
    metrics = compute_metrics(y_test, y_pred, task="multiclass", labels=list(range(NUM_CLASSES)))
    save_metrics(metrics, exp_dir / "metrics.json")
    cm = np.array(metrics["confusion_matrix"])
    save_confusion_matrix(cm, exp_dir / "confusion_matrix.npy", path_png=exp_dir / "confusion_matrix.png", class_names=CLASS_NAMES)
    print("  Test weighted F1: %.4f" % metrics["weighted_f1"])
    return metrics

In [ ]:
# A1: Autoencoder + MLP (load autoencoder-reduced features)
print("=== A1: Autoencoder + MLP ===")
red_dir = get_reduced_dir() / "autoencoder"
X_train_a1 = np.load(red_dir / "train.npy").astype(np.float32)
X_test_a1 = np.load(red_dir / "test.npy").astype(np.float32)
if (red_dir / "val.npy").exists():
    X_val_a1 = np.load(red_dir / "val.npy").astype(np.float32)
else:
    X_val_a1 = None
print("Loaded autoencoder features: train %s test %s (input_dim=%d)" % (X_train_a1.shape, X_test_a1.shape, X_train_a1.shape[1]))
metrics_a1 = train_mlp_and_evaluate(X_train_a1, X_val_a1, X_test_a1, y_train, y_val, y_test, X_train_a1.shape[1], "A1_autoencoder_mlp")

In [ ]:
# B1: Pooling + MLP (load pooled features)
print("=== B1: Pooling + MLP ===")
red_dir = get_reduced_dir() / "pooled"
X_train_b1 = np.load(red_dir / "train.npy").astype(np.float32)
X_test_b1 = np.load(red_dir / "test.npy").astype(np.float32)
if (red_dir / "val.npy").exists():
    X_val_b1 = np.load(red_dir / "val.npy").astype(np.float32)
else:
    X_val_b1 = None
print("Loaded pooled features: train %s test %s (input_dim=%d)" % (X_train_b1.shape, X_test_b1.shape, X_train_b1.shape[1]))
metrics_b1 = train_mlp_and_evaluate(X_train_b1, X_val_b1, X_test_b1, y_train, y_val, y_test, X_train_b1.shape[1], "B1_pooling_mlp")

In [ ]:
# Summary
print("=== Summary ===")
print("A1 (autoencoder + MLP) weighted F1: %.4f" % metrics_a1["weighted_f1"])
print("B1 (pooling + MLP) weighted F1: %.4f" % metrics_b1["weighted_f1"])